# Witches' Brew — Colab runner

Reproduces the CIFAR-10 experiments of Geiping et al. (ICLR 2021) on a Colab GPU.

**Before starting:** Runtime → Change runtime type → GPU. A T4 is enough; one case of the full
protocol (brewing + 8 victim trainings of 40 epochs) takes roughly 1.5–3 h, so run a couple of
cases per session and let the resume logic accumulate them across sessions.

Everything is written to Google Drive, so results and the cached surrogate survive a disconnect.

In [ ]:
# 1. GPU check and Drive mount
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/witches_brew'
!mkdir -p {WORK}

In [ ]:
# 2. Get the project onto the machine.
# Either upload the witches_brew folder to Drive once and point PROJECT at it,
# or clone it from your own git remote.
PROJECT = f'{WORK}/code'   # <- the folder containing pyproject.toml

import os
assert os.path.exists(f'{PROJECT}/pyproject.toml'), 'upload the project folder to Drive first'
%cd {PROJECT}
!pip install -q -e . 2>&1 | tail -2
!python -m pytest tests/ -q 2>&1 | tail -3

In [ ]:
# 3. Keep results, checkpoints and poisons on Drive; keep the dataset on local disk (faster).
RESULTS = f'{WORK}/results'
CKPT = f'{WORK}/checkpoints'
POISONS = f'{WORK}/poisons'
!mkdir -p {RESULTS} {CKPT} {POISONS}

COMMON = (f'--set results_dir={RESULTS} --set checkpoint_dir={CKPT} '
          f'--set poison_dir={POISONS} --set data_path=/content/data --set workers=2')
print(COMMON)

Note: paths and worker counts are infrastructure, not protocol, but they still go through
`--set`, so every result file will list them under `overrides` and report
`paper_setting: false`. Only treat a run as a reproduction when the *scientific* fields
(budget, eps, restarts, steps, epochs, cases, victim_runs) are untouched — those are the ones
`docs/DESIGN_DECISIONS.md` pins to the paper.

In [ ]:
# 4. Smoke test first: whole pipeline, a few minutes.
!python scripts/run_experiment.py --config smoke_cpu.yaml {COMMON} --set device=cuda --set workers=2

In [ ]:
# 5. The paper baseline, two cases at a time (paper: 91.25% +-6.14 over all ten).
#    Re-run with case_ids=2,3 and so on; finished cases are skipped automatically.
!python scripts/run_experiment.py --config cifar10/baseline.yaml {COMMON} --set case_ids=0,1

In [ ]:
# 6. Table 1: differentiable augmentation vs an 8-model ensemble.
for cfg in ['table1_K1_noaug_victim_noaug', 'table1_K1_noaug_victim_aug',
            'table1_K1_diffaug_victim_aug', 'table1_K8_noaug_victim_noaug']:
    !python scripts/run_experiment.py --config cifar10/{cfg}.yaml {COMMON} --set case_ids=0,1

In [ ]:
# 7. Table 2: swap the objective, keep the framework (gradient matching / Bullseye / Poison Frogs).
for cfg in ['table2_resnet18_gm', 'table2_resnet18_bullseye', 'table2_resnet18_frogs']:
    !python scripts/run_experiment.py --config cifar10/{cfg}.yaml {COMMON} --set case_ids=0,1

In [ ]:
# 8. Defenses (section 5.3): differential privacy sweep, then feature-space filtering.
!python scripts/run_defense.py dp --config defenses/dp_sweep.yaml {COMMON} --set case_ids=0
!python scripts/run_defense.py filtering --config defenses/filtering.yaml {COMMON} --set cases=1

In [ ]:
# 9. Inspect the poisons -- the paper's Figs. 6-8 claim they are barely distinguishable.
!python scripts/brew_case.py --config cifar10/baseline.yaml --case 0 --export png {COMMON}

import glob
import matplotlib.pyplot as plt
from PIL import Image

clean = sorted(glob.glob(f'{POISONS}/*case00/clean/*.png'))[:5]
poisoned = sorted(glob.glob(f'{POISONS}/*case00/poisoned/*.png'))[:5]
fig, axes = plt.subplots(2, len(clean), figsize=(2 * len(clean), 4.2))
for col, (a, b) in enumerate(zip(clean, poisoned)):
    axes[0, col].imshow(Image.open(a)); axes[0, col].axis('off')
    axes[1, col].imshow(Image.open(b)); axes[1, col].axis('off')
axes[0, 0].set_title('clean', loc='left')
axes[1, 0].set_title('poisoned', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# 10. Compare against the published numbers.
!python scripts/aggregate.py --results {RESULTS}

## Sweeps and figures

The sweep figures are expensive — each grid point is a full experiment. Start with two cases per
point to see the trend, then extend.

```bash
# section 5.1 inset: restarts x ensemble size
python scripts/sweep.py --config cifar10/inset_RK_sweep.yaml --grid restarts=1,8 ensemble=1,2,4 --set case_ids=0,1
python scripts/plot.py sweep --file results/inset_RK_sweep_sweep.json --x ensemble --series restarts

# Fig. 12 left: thin ResNets, cosine vs Euclidean (the section 3.3 Remark)
python scripts/sweep.py --config cifar10/fig12_width.yaml --grid width=4,8,16,64 objective=cosine,euclidean

# Figs. 2, 5, 9: gradient alignment during victim training
python scripts/run_experiment.py --config cifar10/baseline.yaml --set diagnostics=true --set case_ids=0 --set victim_runs=1 --set name=diag
python scripts/plot.py alignment --case results/diag/case00.json
```

`docs/EXPERIMENTS.md` has the full list with the paper's expected value for each one.